In [84]:
import operator
import os
import re
from typing import Annotated, List, Literal

import wikipedia
from ddgs import DDGS
from dotenv import load_dotenv
from IPython.display import Image, display
from pydantic import BaseModel, Field

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Send

load_dotenv()  

True

In [85]:
### llm setup
llm = ChatGroq(model="openai/gpt-oss-120b")
llm.invoke("hi").content

'Hello! How can I help you today?'

In [ ]:
### Retrievers from first to end
urls=[
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2004.04906v3.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2005.11401v4.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2212.10496v1.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2212.10511v4.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2305.06983v2.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2310.11511v1.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2312.10997v5.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2401.15884v3.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2401.18059v1.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2403.14403v2.pdf"
]

pdfs =[]
for pdf in urls:
    load = PyPDFLoader(pdf).load()
    pdfs.extend(load)
pdfs

In [ ]:
embeddings = HuggingFaceEmbeddings()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
def VS_creation(whole):
    splitter = RecursiveCharacterTextSplitter(chunk_size =1000,chunk_overlap=150)
    chunks = splitter.split_documents(whole)
    vector_s = Chroma.from_documents(chunks ,embeddings)
    return vector_s.as_retriever()

In [ ]:
vs = VS_creation(pdfs)

In [ ]:
vs.invoke("what is RAG?")

[Document(id='250ab63d-5dfe-4f9d-8ce4-84d6b62939eb', metadata={'source': 'D:\\d\\NLP\\Adaptive_Jev_RAG\\pdf\\2312.10997v5.pdf', 'author': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creator': 'LaTeX with hyperref', 'page': 0, 'total_pages': 21, 'page_label': '1', 'trapped': '/False', 'producer': 'pdfTeX-1.40.25', 'title': '', 'creationdate': '2024-03-28T00:54:45+00:00', 'keywords': '', 'moddate': '2024-03-28T00:54:45+00:00', 'subject': ''}, page_content='weaknesses of various approaches in their respective contexts,\nand speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024'),
 Document(id='3dfa19cb-4246-46d5-ae2f-d3aa1782564f', metadata={'source': 'D:\\d\\NLP\\Adaptive_Jev_RAG\\pdf\\2

In [ ]:
from langchain_core.documents import Document
ROUTES = ["direct_answer", "web_search", "rag_retrieval"]

class RAGState(BaseModel):
    question: str = ""
    messages: Annotated[List[AnyMessage], add_messages] = Field(default_factory=list)
    documents: List[Document] = Field(default_factory=list)  # or Annotated[..., operator.add] for parallel retrievers
    decision: Literal["direct_answer", "web_search", "rag_retrieval"] | None = None
    confidence: float = 0.0
    feedback: str = ""
    generation: str = ""
    retries: int = 0


In [ ]:
def retrieve_node(state:RAGState)->dict:
    question = state.question
    result = vs.invoke(question)
    return{"documents":result}

In [ ]:
retrieve_node(RAGState(question="what is the transformers architecture about"))


{'documents': [Document(id='2674be14-a880-4c34-af6a-d512e3104c6d', metadata={'source': 'D:\\d\\NLP\\Adaptive_Jev_RAG\\pdf\\2005.11401v4.pdf', 'keywords': '', 'total_pages': 19, 'moddate': '2021-04-13T00:48:38+00:00', 'creator': 'LaTeX with hyperref', 'trapped': '/False', 'page_label': '2', 'producer': 'pdfTeX-1.40.21', 'title': '', 'page': 1, 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': ''}, page_content='ers Library [66] and can be found at https://github.com/huggingface/transformers/blob/master/\nexamples/rag/. An interactive demo of RAG models can be found at https://huggingface.co/rag/\n2'),
  Document(id='fe6a156f-f6fe-4fb7-bc54-a46c62a30a6f', metadata={'source': 'D:\\d\\NLP\\Adaptive_Jev_RAG\\pdf\\2005.11401v4.pdf', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'total_pages': 19, 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-1

In [ ]:
GRADER_PROMPT = SystemMessage(content="""You are a relevance grader in an Adaptive RAG pipeline.
A router has already judged the user's QUESTION complex enough to need the internal knowledge base of research papers.
You will receive that QUESTION and one retrieved DOCUMENT chunk from the knowledge base.

Decide whether the document contains information that helps answer the question.
Your grades decide the next step: relevant chunks are passed to the answer generator,
and if too few chunks are relevant, the pipeline adapts (rewrites the query or falls back to web search).

Grade "Yes" if the chunk:
- directly answers the question, or part of it, or
- contains facts, definitions, methods, or results that are clearly about the question's topic.

Grade "No" if the chunk:
- is only loosely related (shares a keyword but not the meaning),
- is a reference list, author list, acknowledgements, table of contents, or other boilerplate,
- is about a different concept than the one asked about.

Be strict: the goal is to filter out irrelevant chunks, not to answer the question.
Judge only the given chunk; do not use outside knowledge to fill gaps.""")


In [ ]:
REWRITE_PROMPT = SystemMessage(content="""You are a query rewriter in an Adaptive RAG pipeline.
A router sent the user's QUESTION to the internal knowledge base of research papers, but too few retrieved chunks were relevant.
Your rewritten query will be used for the next attempt: either retrieving again from the knowledge base or searching the web.

Rewrite the question into a single, concise search query that is most likely to return relevant, factual results.

Guidelines:
- Keep the original intent and all key entities, names, acronyms, and technical terms.
- Expand ambiguous acronyms when it helps (e.g. "RAG" -> "retrieval-augmented generation").
- Use the vocabulary a research paper on the topic would use (method names, task names, standard terminology).
- Remove filler words, greetings, and conversational phrasing.
- Prefer keywords and specific terms over full sentences; about 5-15 words.
- Do not answer the question, add facts, or introduce assumptions not present in it.
- Output only the rewritten query, with no explanation or quotes.""")


In [ ]:
class RewrittenQuery(BaseModel):
    """Web-search-optimized version of the user's question."""
    query: str = Field(description="The rewritten web search query")

rewriter = llm.with_structured_output(RewrittenQuery)
def transform_query_node(state: RAGState) -> dict:
    result = rewriter.invoke([
        REWRITE_PROMPT,
        HumanMessage(content=f"QUESTION:\n{state.question}"),
    ])
    return {"question": result.query}


In [ ]:
GENERATION_PROMPT = SystemMessage(content="""You are the answer generator in an Adaptive RAG pipeline about retrieval-augmented generation and LLM research.
A router judged the user's QUESTION too complex to answer from model knowledge alone, so it was sent to retrieval.
You will receive the QUESTION and a set of CONTEXT passages. Each passage is either a chunk from a research paper in the internal knowledge base or a web search result, and each one is labeled with its source.

Answer the question using only the information in the context.

Guidelines:
- Base every claim on the context; do not add facts from your own knowledge.
- If the context only partly answers the question, answer that part and clearly say what is missing.
- If the context does not contain the answer, say "I could not find this in the provided sources." and do not guess.
- Cite the source of each key claim in square brackets, e.g. [2005.11401v4.pdf, p. 3] or [https://example.com].
- If sources disagree, point out the disagreement instead of picking one silently.
- Since the question was routed as complex, give a well-structured answer: start with a direct answer, then the supporting analysis (comparisons, steps, trade-offs) as the question requires. Use short paragraphs or bullet points when that is easier to read.
- Explain technical terms briefly when they are central to the answer.""")


In [ ]:
class GradeDocument(BaseModel):
    """Relevance grade for one chunk retrieved from the research-paper knowledge base in the Adaptive RAG pipeline."""
    relevant: Literal["Yes", "No"] = Field(
        description="'Yes' if the chunk contains facts, definitions, methods, or results that help answer the question; "
                    "'No' if it is only loosely related, about a different concept, or boilerplate (references, author list, acknowledgements)."
    )

grader = llm.with_structured_output(GradeDocument)

def grade_document(state: RAGState) -> dict:
    relevant_docs = []
    for doc in state.documents:
        result = grader.invoke([
            GRADER_PROMPT,
            HumanMessage(content=f"QUESTION:\n{state.question}\n\nDOCUMENT:\n{doc.page_content}"),
        ])
        if result.relevant == "Yes":
            relevant_docs.append(doc)

    return {
        "documents": relevant_docs,
        "feedback": "Yes" if relevant_docs else "No",
    }

In [ ]:
## web search node
def web_search_node(state: RAGState) -> dict:
    try:
        results = DDGS().text(state.question, max_results=5)
    except Exception as e:
        print(f"Web search failed: {e}")
        return {}  

    web_docs = [
        Document(page_content=r["body"], metadata={"source": r["href"], "title": r["title"]})
        for r in results
    ]
    return {"documents": state.documents + web_docs}


In [ ]:
### generate node

def format_docs(docs: List[Document]) -> str:
    parts = []
    for doc in docs:
        source = os.path.basename(doc.metadata.get("source", "unknown"))
        page = doc.metadata.get("page")
        label = f"{source}, p. {page + 1}" if page is not None else source
        parts.append(f"[{label}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)


def generate_node(state: RAGState) -> dict:
    context = format_docs(state.documents)
    response = llm.invoke([
        GENERATION_PROMPT,
        HumanMessage(content=f"QUESTION:\n{state.question}\n\nCONTEXT:\n{context}"),
    ])
    return {"generation": response.content, "messages": [response]}


In [ ]:
def router_node(state:RAGState)-> Literal["transform_query","generate"]:
    if state.feedback =="Yes":
        return "generate"
    return "transform_query"

In [ ]:
graph = StateGraph(RAGState)
graph.add_node("retrieve",retrieve_node)
graph.add_node("grade_documents",grade_document)
graph.add_node("transform_query",transform_query_node)
graph.add_node("web_search",web_search_node)
graph.add_node("generate",generate_node)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve","grade_documents")
graph.add_conditional_edges("grade_documents",router_node)
graph.add_edge("transform_query","web_search")
graph.add_edge("web_search","generate")
graph.add_edge("generate",END)

graph_builder = graph.compile()

try:
    display(Image(graph_builder.get_graph().draw_mermaid_png(max_retries=3, retry_delay=2.0)))
except Exception:
    print(graph_builder.get_graph().draw_mermaid())  # mermaid.ink unreachable: show the diagram as text

In [ ]:
test_cases = [
    # In the PDFs: should go retrieve -> grade -> generate
    {"question": "What is the difference between the RAG-Sequence and RAG-Token models?", "expect": "local"},  # 2005.11401 RAG
    {"question": "What are reflection tokens in Self-RAG and what are they used for?", "expect": "local"},      # 2310.11511 Self-RAG

    # Not in the PDFs: grader should reject the chunks -> transform_query -> web_search -> generate
    {"question": "Who won the 2022 FIFA World Cup final?", "expect": "web"},
]

for case in test_cases:
    print("=" * 80)
    print("Q:", case["question"], f"(expected: {case['expect']})")
    path = []
    for update in graph_builder.stream({"question": case["question"]}, stream_mode="updates"):
        path.extend(update.keys())
        if "generate" in update:
            final = update["generate"]
    went_web = "web_search" in path
    print("Path:", " -> ".join(path))
    print("Went to web:", went_web, "-> OK" if went_web == (case["expect"] == "web") else "-> WRONG")
    print("Answer:", final["generation"][:500], "...")

Q: What is the difference between the RAG-Sequence and RAG-Token models? (expected: local)
Path: retrieve -> grade_documents -> transform_query -> generate -> web_search -> generate
Went to web: True -> WRONG
Answer: **Answer**

- **RAG‑Sequence**: Retrieves the top‑K documents once for the whole input *x* and then generates the entire answer *y* conditioned on a *single* latent document *z* chosen from that set. The probability of the whole sequence is obtained by marginalising over the K documents **once**, i.e. the same document is used for every token of the output.  

- **RAG‑Token**: Retrieves the same top‑K documents, but for **each output token** it can pick a (different) latent document from the ret ...
Q: What are reflection tokens in Self-RAG and what are they used for? (expected: local)
Path: retrieve -> grade_documents -> transform_query -> generate -> web_search -> generate
Went to web: True -> WRONG
Answer: **Direct answer**

Self‑RAG introduces a small set of *reflectio

#### WEB search normal one


In [ ]:
## web search node
def web_search_node(state: RAGState) -> dict:
    try:
        results = DDGS().text(state.question, max_results=5)
    except Exception as e:
        print(f"Web search failed: {e}")
        return {}  

    web_docs = [
        Document(page_content=r["body"], metadata={"source": r["href"], "title": r["title"]})
        for r in results
    ]
    return {"documents": state.documents + web_docs}


In [ ]:
SYSTEM_PROMPT = SystemMessage(content="""You are a helpful, knowledgeable assistant in an Adaptive RAG system specialised in retrieval-augmented generation and LLM research.
A router has judged the user's message simple enough to answer directly from your own knowledge, without retrieving documents or searching the web.

Guidelines:
- Answer clearly and concisely: give the direct answer first, then a short explanation only if it helps.
- For greetings and small talk, reply briefly and naturally, and offer help.
- For definitions and concepts, give an accurate, plain-language explanation; add a short example when it makes the idea clearer.
- For math or simple reasoning, show the key steps and the final result.
- Reply in the same language the user writes in.
- Be honest about uncertainty: if you are not sure, say so instead of guessing.
- Do not invent facts, numbers, paper titles, citations, or links.
- If the question actually needs up-to-date information (news, prices, recent releases) or detailed analysis of specific papers, say that you cannot answer it reliably from memory and that it needs a web search or the document knowledge base.""")
def llm_answer_node(state: RAGState) -> dict:
    response = llm.invoke([
        SYSTEM_PROMPT,
        HumanMessage(content=state.question),
    ])
    return {"generation": response.content, "messages": [response]}


### JEV


In [ ]:
import torch
from so1 import Decider, Choice

JEV_MODEL = "Qwen/Qwen3.5-4B"
THRESHOLD = 0.95

decider = Decider.from_pretrained(JEV_MODEL, backend="hf", dtype=torch.bfloat16)

ROUTES = ["direct_answer", "web_search", "rag_retrieval"]

ROUTE_DESCRIPTIONS = """You are routing a user query to one of three routes based on how complex it is:
- direct_answer: the query is simple (greetings, basic facts, definitions, simple math). The LLM answers it directly from its own knowledge.
- web_search: the query is a little complex. It needs extra or up-to-date information from the internet (news, prices, recent events, current versions).
- rag_retrieval: the query is very complex. It needs deep, detailed analysis using information retrieved from the internal document knowledge base."""

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [ ]:
def route_query(state:RAGState):
    decision = decider.decide(
        state=f"{ROUTE_DESCRIPTIONS}\n\nUser query: {state.question}",
        questions=[Choice("Which route should handle this query?", ROUTES)],
        mode="separate",
    )[0]
    return decision.choice, decision.confidence

In [ ]:
def jev_node(state: RAGState) -> dict:
    route, confidence = route_query(state)
    if confidence < THRESHOLD:
        route = "rag_retrieval"  
    print(f"Route: {route} ({confidence:.2f})")
    return {"decision": route, "confidence": confidence}


def route_decision(state: RAGState) -> Literal["normal", "web", "RAG"]:
    if state.decision == "direct_answer":
        return "normal"
    if state.decision == "web_search":
        return "web"
    return "RAG"


grapher = StateGraph(RAGState)
grapher.add_node("jev", jev_node)
grapher.add_node("normal", llm_answer_node)
grapher.add_node("web", web_search_node)
grapher.add_node("web_generate", generate_node)   
grapher.add_node("RAG", graph_builder)           

grapher.set_entry_point("jev")
grapher.add_conditional_edges("jev", route_decision)
grapher.add_edge("normal", END)
grapher.add_edge("web", "web_generate")
grapher.add_edge("web_generate", END)
grapher.add_edge("RAG", END)

adaptive_rag = grapher.compile()

try:
    display(Image(adaptive_rag.get_graph().draw_mermaid_png(max_retries=3, retry_delay=2.0)))
except Exception:
    print(adaptive_rag.get_graph().draw_mermaid())  # mermaid.ink unreachable: show the diagram as text

In [ ]:
result = adaptive_rag.invoke({"question": "What are reflection tokens in Self-RAG?"})
print(result["decision"], result["confidence"])
print(result["generation"])
